In [11]:
import sys
sys.path.append('/home/wuct/ALICE/reps/hf-vn-dev/dev-v0/utils/')
from load_utils import list_files
file_path = '/home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/cutVar'
files_path = list_files(file_path, prefix='cutVar', suffix='.root')
print(f'Found {len(files)} files:')
for f in files:
    print(f'  {f}')

import ROOT
a_side_files = []
b_side_files = []
for f_path in files_path:
    if 'a_side' in f_path:
        a_side_files.append(ROOT.TFile(f_path))
    elif 'b_side' in f_path:
        b_side_files.append(ROOT.TFile(f_path))
    else:
        print(f'Warning: file {f_path} does not contain "a_side" or "b_side" in its name, skipping.')
def get_histograms(file_list):
    histograms = []
    for f in file_list:
        hist_c = f.Get('hCorrYieldPrompt')
        hist_b = f.Get('hCorrYieldFD')
        hist = hist_c.Clone()
        hist.Add(hist_b)
        if hist:
            histograms.append(hist)
        else:
            print(f'Warning: histogram "hCorrYieldPrompt" or "hCorrYieldFD" not found in file {f.GetName()}')
    return histograms
a_side_corr_yields = get_histograms(a_side_files)
b_side_corr_yields = get_histograms(b_side_files)

Found 10 files:
  /home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/cutVar/a_side/pt_0_71/cutVar/cutVar.root
  /home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/cutVar/a_side/pt_71_74/cutVar/cutVar.root
  /home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/cutVar/a_side/pt_74_76/cutVar/cutVar.root
  /home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/cutVar/a_side/pt_76_79/cutVar/cutVar.root
  /home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/cutVar/a_side/pt_79_97/cutVar/cutVar.root
  /home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/cutVar/b_side/pt_0_71/cutVar/cutVar.root
  /home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/cutVar/b_side/pt_71_74/cutVar/cutVar.root
  /home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/test2/cutvar_test_combined/cutV

In [26]:
import ROOT
ROOT.gStyle.SetOptStat(0)
canvas_list = []
legend_list = []
canvas_list.append(ROOT.TCanvas('c1', 'c1', 1600, 1200))
markers = [ROOT.kFullCircle, ROOT.kOpenSquare]
colors = [
    # ROOT.kBlack,       
    ROOT.kRed-4,       
    ROOT.kBlue-4, 
    ROOT.kGreen+2,     
    ROOT.kOrange+7,   
    # ROOT.kYellow-7,    
    ROOT.kMagenta-3,   
    ROOT.kCyan-3,      
    ROOT.kSpring-5,    
    ROOT.kViolet-4,    
    ROOT.kTeal-5,    
    ROOT.kGray+1    
]
legend_list.append(ROOT.TLegend(0.5, 0.6, 0.9, 0.9))
legs = ['0 - 0.71 GeV/c', '0.71 - 0.74 GeV/c', '0.74 - 0.76 GeV/c', '0.76 - 0.79 GeV/c', '0.79 - 0.97 GeV/c']
legend_list[0].SetNColumns(3)
legend_list[0].SetBorderSize(0)
legend_list[0].SetFillStyle(0)

max_Y = max(hist.GetMaximum() for hist in a_side_corr_yields + b_side_corr_yields)
# canvas_list[0].SetGrid()
canvas_list[0].SetTitle('; pT; Nc + Nb')
# frame = canvas_list[0].DrawFrame(1, 0, 5, max_Y*1.2)

legend_list[0].AddEntry(0, "", "")              
legend_list[0].AddEntry(0, "A side", "")        
legend_list[0].AddEntry(0, "B side", "")         
for i, (hist_a, hist_b) in enumerate(zip(a_side_corr_yields, b_side_corr_yields)):
    hist_a.SetStats(0)
    hist_a.SetMarkerStyle(markers[0])
    hist_a.SetMarkerColor(colors[i])
    hist_a.SetLineColor(colors[i])
    hist_a.SetLineWidth(2)
    hist_a.SetMarkerSize(3)
    # hist_b.SetStats(0)
    hist_b.SetMarkerStyle(markers[1])
    hist_b.SetMarkerColor(colors[i])
    hist_b.SetLineColor(colors[i])
    hist_b.SetLineWidth(2)
    hist_b.SetMarkerSize(3)
    if i == 0:
        hist_a.SetMinimum(0)
        hist_a.SetMaximum(120000)
        hist_a.Draw('PE')
        hist_b.Draw('PE SAME')
    else:
        hist_a.Draw('PE SAME')
        hist_b.Draw('PE SAME')
    legend_list[0].AddEntry(0, legs[i], "")
    legend_list[0].AddEntry(hist_a, "", "pe")
    legend_list[0].AddEntry(hist_b, "", "pe")
legend_list[0].Draw()
canvas_list[0].SaveAs('charm_bulk_correlation_yields.png')

Info in <TCanvas::Print>: png file charm_bulk_correlation_yields.png has been created


In [31]:
import ROOT
ROOT.gStyle.SetOptStat(0)

canvas_list = []
legend_list = []

# 画布高度设大一点，容纳上下两个图
canvas_list.append(ROOT.TCanvas('c1', 'c1', 1600, 1400))

markers = [ROOT.kFullCircle, ROOT.kOpenSquare]
colors = [
    ROOT.kRed-4, ROOT.kBlue-4, ROOT.kGreen+2, ROOT.kOrange+7, 
    ROOT.kMagenta-3, ROOT.kCyan-3, ROOT.kSpring-5, 
    ROOT.kViolet-4, ROOT.kTeal-5, ROOT.kGray+1 
]

# ==========================================
# 1. 切分画布为上下两个 Pad
# ==========================================
# 上半部分：主图 (占 70%)
pad1 = ROOT.TPad("pad1", "pad1", 0, 0.3, 1, 1.0)
pad1.SetBottomMargin(0.02) # 去除上下图缝隙
pad1.Draw()

# 下半部分：Ratio 图 (占 30%)
pad2 = ROOT.TPad("pad2", "pad2", 0, 0.0, 1, 0.3)
pad2.SetTopMargin(0.02)
pad2.SetBottomMargin(0.3) # 留空间给 X 轴标题
pad2.Draw()

# ==========================================
# 2. 绘制上半部分 (主图)
# ==========================================
pad1.cd()

legend_list.append(ROOT.TLegend(0.45, 0.55, 0.9, 0.9))
legs = ['0 - 0.71 GeV/c', '0.71 - 0.74 GeV/c', '0.74 - 0.76 GeV/c', '0.76 - 0.79 GeV/c', '0.79 - 0.97 GeV/c']
legend_list[0].SetNColumns(3)
legend_list[0].SetBorderSize(0)
legend_list[0].SetFillStyle(0)

legend_list[0].AddEntry(0, "", "")              
legend_list[0].AddEntry(0, "A side", "")        
legend_list[0].AddEntry(0, "B side", "")   

for i, (hist_a, hist_b) in enumerate(zip(a_side_corr_yields, b_side_corr_yields)):
    hist_a.SetStats(0)
    hist_a.SetMarkerStyle(markers[0])
    hist_a.SetMarkerColor(colors[i])
    hist_a.SetLineColor(colors[i])
    hist_a.SetLineWidth(2)
    hist_a.SetMarkerSize(3)
    
    hist_b.SetMarkerStyle(markers[1])
    hist_b.SetMarkerColor(colors[i])
    hist_b.SetLineColor(colors[i])
    hist_b.SetLineWidth(2)
    hist_b.SetMarkerSize(3)
    
    if i == 0:
        hist_a.SetMinimum(0)
        hist_a.SetMaximum(120000)
        
        # 隐藏 X 轴标签，移交到底部图
        hist_a.GetXaxis().SetLabelSize(0)
        hist_a.GetXaxis().SetTitleSize(0)
        hist_a.GetYaxis().SetTitle('N_{c} + N_{b}')
        
        hist_a.Draw('PE')
        hist_b.Draw('PE SAME')
    else:
        hist_a.Draw('PE SAME')
        hist_b.Draw('PE SAME')
        
    legend_list[0].AddEntry(0, legs[i], "")
    legend_list[0].AddEntry(hist_a, "", "pe")
    legend_list[0].AddEntry(hist_b, "", "pe")
    
legend_list[0].Draw()

# ==========================================
# 3. 绘制下半部分 (Ratio)
# ==========================================
pad2.cd()

# 提取分母：A侧的分母是 A[0], B侧的分母是 B[0]
denom_a = a_side_corr_yields[0].Clone("denom_a")
denom_b = b_side_corr_yields[0].Clone("denom_b")

# 防止垃圾回收
ratio_a_list = []
ratio_b_list = []

for i, (hist_a, hist_b) in enumerate(zip(a_side_corr_yields, b_side_corr_yields)):
    # 克隆当前直方图作为分子
    ratio_a = hist_a.Clone(f"ratio_a_{i}")
    ratio_b = hist_b.Clone(f"ratio_b_{i}")
    if i == 0:
        continue
    # 核心：A 除以 A 的 0-0.71，B 除以 B 的 0-0.71
    ratio_a.Divide(denom_a)
    ratio_b.Divide(denom_b)
    
    if i == 1:
        ratio_a.SetTitle("")
        
        # --- Ratio 图 Y 轴设置 ---
        ratio_a.GetYaxis().SetTitle("Ratio") # 标题简化
        ratio_a.GetYaxis().SetRangeUser(0.5, 3) # 请根据数据范围微调这里
        ratio_a.GetYaxis().SetNdivisions(505)
        
        # 放大字号以适应扁平的 Pad
        ratio_a.GetYaxis().SetTitleSize(0.1)
        ratio_a.GetYaxis().SetTitleOffset(0.4)
        ratio_a.GetYaxis().SetLabelSize(0.08)
        
        # --- Ratio 图 X 轴设置 ---
        ratio_a.GetXaxis().SetTitle("p_{T} (GeV/c)")
        ratio_a.GetXaxis().SetTitleSize(0.12)
        ratio_a.GetXaxis().SetTitleOffset(1.0)
        ratio_a.GetXaxis().SetLabelSize(0.1)
        
        ratio_a.Draw("PE")
    else:
        ratio_a.Draw("PE SAME")
        
    ratio_b.Draw("PE SAME")
    
    ratio_a_list.append(ratio_a)
    ratio_b_list.append(ratio_b)

# 画一条 Y=1 的虚线
pad2.Update()
line = ROOT.TLine(pad2.GetUxmin(), 1.0, pad2.GetUxmax(), 1.0)
line.SetLineStyle(2)
line.SetLineColor(ROOT.kBlack)
line.Draw()

canvas_list[0].SaveAs('charm_bulk_correlation_yields_with_ratio.png')

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: png file charm_bulk_correlation_yields_with_ratio.png has been created


In [33]:
import ROOT
ROOT.gStyle.SetOptStat(0)

# 定义要绘制的数据组，格式为：(标题, 数据集, 输出文件名)
datasets = [
    ("A side", a_side_corr_yields, 'charm_bulk_correlation_yields_Aside_ratio.png'),
    ("B side", b_side_corr_yields, 'charm_bulk_correlation_yields_Bside_ratio.png')
]

colors = [
    ROOT.kRed-4, ROOT.kBlue-4, ROOT.kGreen+2, ROOT.kOrange+7, 
    ROOT.kMagenta-3, ROOT.kCyan-3, ROOT.kSpring-5, 
    ROOT.kViolet-4, ROOT.kTeal-5, ROOT.kGray+1 
]
legs = ['0 - 0.71 GeV/c', '0.71 - 0.74 GeV/c', '0.74 - 0.76 GeV/c', '0.76 - 0.79 GeV/c', '0.79 - 0.97 GeV/c']

# 用于防止 ROOT 对象被 Python 垃圾回收的全局列表
canvas_list = []
ratio_list = [] 

# 开始循环：第一次画 A，第二次画 B
for side_name, yields, out_name in datasets:
    
    # 既然分开了，宽度可以从 1600 稍微收窄一点，比如 1200
    c = ROOT.TCanvas(f'c_{side_name}', side_name, 1200, 1400)
    
    # ==========================================
    # 1. 切分画布
    # ==========================================
    pad1 = ROOT.TPad(f"pad1_{side_name}", "pad1", 0, 0.3, 1, 1.0)
    pad1.SetBottomMargin(0.02)
    pad1.Draw()

    pad2 = ROOT.TPad(f"pad2_{side_name}", "pad2", 0, 0.0, 1, 0.3)
    pad2.SetTopMargin(0.02)
    pad2.SetBottomMargin(0.3)
    pad2.Draw()

    # ==========================================
    # 2. 绘制上半部分 (主图)
    # ==========================================
    pad1.cd()
    
    # 图例恢复为标准的单列，并加上 Header 标明 A/B
    legend = ROOT.TLegend(0.5, 0.55, 0.9, 0.9)
    legend.SetHeader(side_name, "C") # "C" 表示标题居中
    legend.SetBorderSize(0)
    legend.SetFillStyle(0)

    # 动态获取当前数据集的最大值，防止 B side 的数据超限
    max_Y = max(hist.GetMaximum() for hist in yields)

    for i, hist in enumerate(yields):
        hist.SetStats(0)
        hist.SetMarkerStyle(ROOT.kFullCircle) # 统一全部使用实心点
        hist.SetMarkerColor(colors[i])
        hist.SetLineColor(colors[i])
        hist.SetLineWidth(2)
        hist.SetMarkerSize(3)
        
        if i == 0:
            hist.SetMinimum(0)
            hist.SetMaximum(120000) # 留出 30% 空间给图例
            
            hist.GetXaxis().SetLabelSize(0)
            hist.GetXaxis().SetTitleSize(0)
            hist.GetYaxis().SetTitle('N_{c} + N_{b}')
            
            hist.Draw('PE')
        else:
            hist.Draw('PE SAME')
            
        # 直接把当前直方图加入图例
        legend.AddEntry(hist, legs[i], "pe")
        
    legend.Draw()

    # ==========================================
    # 3. 绘制下半部分 (Ratio)
    # ==========================================
    pad2.cd()
    denom = yields[0].Clone(f"denom_{side_name}")
    
    current_ratios = [] # 暂存当前的 ratio 图
    
    for i, hist in enumerate(yields):
        if i == 0:
            continue
            
        ratio = hist.Clone(f"ratio_{side_name}_{i}")
        ratio.Divide(denom)
        
        if i == 1: # 因为 i=0 被跳过了，所以 i=1 是第一张要画的图
            ratio.SetTitle("")
            
            ratio.GetYaxis().SetTitle("Ratio") 
            ratio.GetYaxis().SetRangeUser(0.5, 3) 
            ratio.GetYaxis().SetNdivisions(505)
            
            ratio.GetYaxis().SetTitleSize(0.1)
            ratio.GetYaxis().SetTitleOffset(0.4)
            ratio.GetYaxis().SetLabelSize(0.08)
            
            ratio.GetXaxis().SetTitle("p_{T} (GeV/c)")
            ratio.GetXaxis().SetTitleSize(0.12)
            ratio.GetXaxis().SetTitleOffset(1.0)
            ratio.GetXaxis().SetLabelSize(0.1)
            
            ratio.Draw("PE")
        else:
            ratio.Draw("PE SAME")
            
        current_ratios.append(ratio)
        
    pad2.Update()
    line = ROOT.TLine(pad2.GetUxmin(), 1.0, pad2.GetUxmax(), 1.0)
    line.SetLineStyle(2)
    line.SetLineColor(ROOT.kBlack)
    line.Draw()
    
    # 防止 Python 垃圾回收机制把 ROOT 的指针清理掉
    c.legend = legend
    c.line = line
    canvas_list.append(c)
    ratio_list.append(current_ratios)
    
    # 保存当前的画布
    c.SaveAs(out_name)

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_A side
Info in <TCanvas::Print>: png file charm_bulk_correlation_yields_Aside_ratio.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_B side
Info in <TCanvas::Print>: png file charm_bulk_correlation_yields_Bside_ratio.png has been created
